In [ ]:
dbutils.widgets.text("ticker", "VOD")
dbutils.widgets.text("mic", "XLON")
dbutils.widgets.text("start_date", "2022-11-04")

ticker = dbutils.widgets.get("ticker")
mic = dbutils.widgets.get("mic")
start_date = dbutils.widgets.get("start_date")

## BMLL Notebook Educational Series
### Transaction Cost Analysis (TCA) for European Markets

### Understanding Transaction Cost Analysis

Transaction Cost Analysis (TCA) has become essential in modern finance for evaluating execution quality and trading performance. TCA decomposes the total cost of a trade into measurable components, allowing portfolio managers and traders to:

- **Evaluate broker and algorithm performance** - Compare execution prices against market benchmarks
- **Identify execution inefficiencies** - Detect systematic slippage and adverse market impact
- **Optimize trading strategies** - Understand cost drivers and refine order routing decisions
- **Benchmark against alternatives** - Measure performance across venues, time periods, and security characteristics

> "Transaction cost analysis (TCA) has regained a newfound interest in the financial community as a result of the proliferation of algorithmic trading. Portfolio managers and traders are using TCA to evaluate the performance of brokers and their algorithms."
— *Robert Kissell, The Science of Algorithmic Trading and Portfolio Management, 2014*

### European Trade-Level TCA Framework

This notebook implements a comprehensive TCA framework for European equity markets using BMLL's high-quality market data. The analysis captures three key TCA dimensions:

**1. Arrival Price (CBBO at Trade Time)**
- The consolidated best bid/offer (CBBO) price at the exact moment of trade execution
- Represents the "fair market value" at execution time
- Used as the benchmark to measure execution quality
- Accounts for multi-venue pricing across all European trading venues

**2. Markouts (Post-Trade Price Movement)**
- Price movement of the CBBO midpoint at fixed time intervals (1ms, 10ms, 100ms, 1s, 5s)
- Measured in basis points (1 bp = 0.01%)
- Indicates market impact: how much the market moved against the trade immediately after execution
- Positive markouts (prices moved in execution direction) are favorable
- Negative markouts (prices moved against execution direction) indicate market impact costs

**3. Close Price & Benchmarks**
- Closing price from the primary listing venue's closing auction (or last continuous trade price if no auction volume)
- VWAP (Volume-Weighted Average Price) - benchmark for algorithmic trades with extended durations
- TWAP (Time-Weighted Average Price) - benchmark for passive positioning strategies

Together, these metrics provide a complete picture of execution performance from the moment a trade executes through market close.

### How to Interpret Results

For each trade execution, you will receive:
- **Execution Price**: The price at which your order executed
- **Arrival Price**: The CBBO midpoint at execution time (your benchmark)
- **Markout @ 1s/5s**: Price movement immediately after execution (market impact measurement)
- **Close Price**: Day-end price for assessing longer-term performance
- **VWAP/TWAP**: Algorithmic trading benchmarks for extended order durations

A well-executed trade will have:
- Small difference between execution price and arrival price (tight execution)
- Positive markouts (prices moved in your favor post-execution)
- Overall cost at or below VWAP for the period

In [ ]:
import numpy as np
import plotly.express as px
import pandas as pd

pd.options.display.max_columns = None

## Data Retrieval and Processing Functions

This section implements the core helper functions for computing TCA metrics. These functions handle:
- **Instrument identification** - Mapping tickers and MICs to BMLL instrument identifiers
- **Market data retrieval** - Fetching CBBO quotes, trade data, and closing prices
- **Markout calculation** - Computing post-trade price movements at multiple time horizons
- **Benchmark computation** - Computing VWAP and TWAP prices for trade comparison

Platform note: These functions adapt to notebook (BMLL API), Snowflake, or Databricks environments
through conditional logic in their implementations.

### Instrument Data Lookup

The first step in TCA is mapping external identifiers (ticker + MIC) to BMLL's internal instrument IDs.
This allows us to:
1. Query the correct market data for the security
2. Access all venues where the instrument trades
3. Retrieve reference information (listings, primary exchange, etc.)

NOTE: This function queries a database and should not be run in parallel.

### Markout Calculation

Markouts measure how much the market moved immediately after a trade execution.
Positive markouts indicate the market moved in the execution direction (favorable).
Negative markouts indicate adverse price movement (market impact).

**Calculation steps:**
1. For each trade, identify execution timestamp and execution price
2. For each markout time horizon (1ms, 10ms, 100ms, 1s, 5s):
   - Find the CBBO midpoint at (execution_time + horizon)
   - Calculate price change: (future_price - execution_price) / execution_price
   - Convert to basis points: change × 10,000 bps
3. Apply sign correction: BUY orders (positive) record gains when price increases;
   SELL orders (negative) need sign reversal to show gains when price decreases

The result is a set of columns showing how price movement unfolded at each time horizon.

In [ ]:
def get_ebbo_markouts(
    markout_quotes, mark_outs, name="ebbo", execution_price="execution_price"
):
    """
    Calculate post-trade markouts across multiple time horizons.

    For each trade execution, computes how far the CBBO midpoint has moved at fixed time
    intervals (1ms through 5s). Markouts are expressed in basis points and adjusted for
    trade direction (buys/sells) to show gains/losses consistently.

    Parameters:
        markout_quotes (pd.DataFrame): Trade/execution data with 'event_timestamp' (start_time),
                                       'execution_price', and 'side' columns
        mark_outs (pd.DataFrame): CBBO time series with 'event_timestamp' and 'mid_price'
        name (str): Label for columns (e.g., 'ebbo' produces 'markout_to_ebbo_mid_1ms')
        execution_price (str): Column name containing execution prices

    Returns:
        pd.DataFrame: Original markout_quotes columns plus markout columns for each time horizon:
                     'markout_to_{name}_mid_{interval}' (in basis points)
                     - Positive values: prices moved in execution direction (favorable)
                     - Negative values: prices moved against execution (market impact cost)

    Implementation notes:
        - Uses merge_asof to efficiently locate quotes near target timestamps
        - Handles missing data (NaN quotes) appropriately
        - Reverses markout signs for sell orders (side==2) so gains are always positive
    """

    # Step 1: Prepare timestamps and column tracking
    # Save original columns so we only return relevant output columns
    df_cols = list(markout_quotes.columns)
    markout_times = ("1ms", "10ms", "100ms", "1s", "5s")

    # Standardize timestamp column names for the merge operations
    markout_quotes["event_timestamp"] = pd.to_datetime(markout_quotes["start_time"])
    mark_outs["event_timestamp"] = pd.to_datetime(mark_outs["event_timestamp"])
    mark_outs[f"{name}_midpoint"] = mark_outs["mid_price"]

    time_cols = []  # Will hold target timestamp columns (e.g., ts_1ms, ts_100ms)
    res_cols = []   # Will hold result columns for each markout time horizon

    # Step 2: Create future timestamp columns for each markout horizon
    # These represent the times when we want to measure the CBBO midpoint
    # e.g., if execution at 10:00:00.000, ts_100ms = 10:00:00.100
    for time in markout_times:
        markout_quotes["ts_" + time] = markout_quotes["event_timestamp"] + pd.Timedelta(time)
        time_cols.append("ts_" + time)

    # Step 3: For each markout horizon, join the CBBO midpoint at that time
    # Uses merge_asof to efficiently find the nearest CBBO quote at or after the target timestamp
    for col in time_cols:
        # Sort by target timestamp column to prepare for merge_asof
        markout_quotes = markout_quotes.sort_values(col)

        # merge_asof finds the first CBBO quote at or after the target time
        # This gives us the midpoint at that markout horizon
        markout_quotes = pd.merge_asof(
            left=markout_quotes,
            right=mark_outs[["event_timestamp", f"{name}_midpoint"]],
            right_on="event_timestamp",
            left_on=col,
            suffixes=("", "_x"),
        )

        # Rename the joined column to indicate which time horizon it represents
        markout_quotes = markout_quotes.rename(
            {f"{name}_midpoint": f"{name}_midpoint_{col}"}, axis=1
        ).drop("event_timestamp_x", axis=1)

        # Clean up rows with zero timestamps (indicates no match found)
        markout_quotes.loc[markout_quotes[col] == pd.to_datetime(0), col] = np.nan
        markout_quotes.loc[
            markout_quotes[col] == pd.to_datetime(0), f"{name}_midpoint_{col}"
        ] = np.nan

    # Reset index for cleaner output
    markout_quotes = markout_quotes.sort_values("event_timestamp").reset_index(drop=True)

    # Step 4: Calculate markout in basis points for each time horizon
    # Markout = (future_price - execution_price) / execution_price * 10,000 bps
    # This shows how far the market moved from the execution price
    for time in markout_times:
        # Initialize column with NaN (will be filled below)
        markout_quotes[f"markout_to_{name}_mid_{time}"] = np.nan

        # Calculate markout only where execution_price is available
        markout_quotes.loc[
            ~markout_quotes.execution_price.isna(), f"markout_to_{name}_mid_{time}"
        ] = (
            (
                markout_quotes[f"{name}_midpoint_ts_{time}"]
                - markout_quotes[execution_price]
            )
            / markout_quotes[execution_price]
            * 10000
        )

        res_cols.append(f"markout_to_{name}_mid_{time}")

    # Step 5: Adjust markout signs for sell orders (side==2)
    # For BUY orders (side==1): positive markout means the market moved up (favorable)
    # For SELL orders (side==2): we want negative markout to mean price moved down (favorable)
    # So we reverse the sign for sell orders to make positive markouts always favorable
    markout_quotes.loc[markout_quotes['side']=='2', res_cols] *= -1

    # Return only the original input columns plus the newly computed markout columns
    return markout_quotes[df_cols + res_cols]

### Benchmark Price Calculation

VWAP and TWAP are standard benchmarks used to evaluate algorithmic trading performance:
- **TWAP (Time-Weighted Average Price)**: For passive positioning/market-making strategies
- **VWAP (Volume-Weighted Average Price)**: For algorithms that participate in market volume

A well-executed trade should have costs at or below these benchmarks.

In [ ]:
def get_twap_value(ebbo, start_ts, end_ts):
    """
    Calculate time-weighted average price (TWAP) for a time window.

    Computes the average CBBO midpoint weighted by duration at each price level.
    Formula: sum(price × time_duration) / sum(time_duration)

    TWAP is used to benchmark passive strategies (e.g., market-making, liquidity provision)
    where the trader's objective is to capture bid-ask spread rather than move prices.

    Parameters:
        ebbo (pd.DataFrame): CBBO time series with 'event_timestamp' and 'mid_price'
        start_ts (str/timestamp): Window start time
        end_ts (str/timestamp): Window end time

    Returns:
        float: Time-weighted average price for the [start_ts, end_ts] window

    Implementation note:
        - Clips timestamps to window boundaries (handles quotes before/after window)
        - Uses negative duration to calculate time spent at each price
        - Handles duration gaps naturally (prices hold until next quote)
    """
    
    middt = ebbo.sort_values("event_timestamp").assign(event_timestamp=lambda x:x["event_timestamp"].clip(
        pd.to_datetime(start_ts), pd.to_datetime(end_ts)
    )).assign(
        dt=lambda x:-x['event_timestamp'].diff(-1).dt.total_seconds())
    
    return (middt['mid_price'] * middt['dt']).sum() / middt['dt'].sum()

In [ ]:
def get_vwap_value(trades, start_ts, end_ts, filter_lit=True):
    """
    Calculate volume-weighted average price (VWAP) for a time window.

    Computes the average execution price weighted by trading volume at each price level.
    Formula: sum(price × volume) / sum(volume)

    VWAP is used to benchmark algorithmic trading algorithms that participate in market
    volume. Well-executed algorithms should achieve execution prices at or below VWAP.

    Parameters:
        trades (pd.DataFrame): Trade time series with 'event_timestamp', 'price', 'execution_size',
                               'bmll_trade_type', and 'market_state'
        start_ts (str/timestamp): Window start time
        end_ts (str/timestamp): Window end time
        filter_lit (bool): If True, includes only lit executions (LIT, UNCROSSING);
                          if False, includes all trade types (lit, dark, etc.)

    Returns:
        float: Volume-weighted average price for the [start_ts, end_ts] window

    Edge cases:
        - Empty window: Returns 0 if no trades in window (edge case, should be filtered)
        - Single trade: Returns that trade's price as weighted average
        - Market state filtering: When filter_lit=True, only trades during
          CONTINUOUS_TRADING, OPENING_AUCTION, or CLOSING_AUCTION are included
    """
    if filter_lit:
        trades = trades.query(
            "bmll_trade_type.isin(['LIT', 'UNCROSSING']).values & market_state.isin(['CONTINUOUS_TRADING', 'OPENING_AUCTION', 'CLOSING_AUCTION']).values"
        )
        
    volprice = trades.sort_values("event_timestamp").reset_index(drop=True)

    volprice = volprice[volprice.event_timestamp.between(pd.to_datetime(start_ts), pd.to_datetime(end_ts))]
    
    return (volprice['execution_size'] * volprice['price']).sum() / volprice['execution_size'].sum()

## TCA Report Generation

### Pipeline Overview

The generate_tca_report function orchestrates all helper functions to produce a complete
TCA analysis for a single instrument. It:

1. **Fetches market data** - CBBO quotes and executed trades for the instrument
2. **Joins arrival prices** - CBBO midpoint at each execution time (execution benchmark)
3. **Calculates markouts** - Post-trade price movements at 5 time horizons
4. **Determines close price** - Day-end reference price (closing auction or last trade)
5. **Computes benchmarks** - VWAP and TWAP for algorithmic trading comparison

The function handles errors gracefully (returns None on failure) and can be parallelized
for batch processing of multiple instruments.

In [ ]:
def generate_tca_report(iid, date):
    """
    Generate comprehensive TCA metrics for a security and date.

    Orchestrates all helper functions to produce a complete TCA report. For each execution
    in the input data, computes:
    - Arrival price (CBBO at execution time)
    - Markouts at 1ms, 10ms, 100ms, 1s, 5s horizons (basis points)
    - Close price (day-end benchmark)
    - VWAP and TWAP (algorithmic trading benchmarks)

    Parameters:
        iid (int): BMLL instrument identifier
        date (str): Trading date in YYYY-MM-DD format

    Returns:
        pd.DataFrame: Original execution data enriched with TCA columns:
                     - arrival_ebb_price, arrival_ebo_price: CBBO at execution
                     - arrival_price: CBBO midpoint (benchmark price)
                     - markout_to_ebbo_mid_*: Post-trade price movements (bps) at each horizon
                     - close_price: Day-end closing price
                     - close_flag: True if auction close, False if last trade price
                     - vwap: Volume-weighted average price for window
                     - twap: Time-weighted average price for window

    Error handling:
        - Returns None on exceptions (instrument not found, missing data, etc.)
        - Exceptions are printed to stdout for debugging
        - Consider catching and logging these exceptions at call site
    """
    try:
        # Step 1: Extract execution records for this instrument
        df_iid = custom_orders[custom_orders.InstrumentId == iid].copy()
        df_iid.start_time = pd.to_datetime(df_iid.start_time)

        # Step 2: Fetch consolidated best bid/offer quotes for the day
        ebbo = get_ebbo(iid, date)

        # Step 3: Join arrival price (CBBO at execution time)
        # Uses merge_asof to find the CBBO quote at the exact execution timestamp
        df_iid = pd.merge_asof(
            df_iid.sort_values("start_time"),
            ebbo[["event_timestamp", "mid_price", "ebb_price", "ebo_price"]].rename(
                columns={"event_timestamp": "start_time",
                         "ebb_price": "arrival_ebb_price",
                         "ebo_price": "arrival_ebo_price"}
            ),
            on="start_time",
            allow_exact_matches=True,
        ).rename(columns={"mid_price": "arrival_price"})

        # Step 4: Add closing price (auction or last trade price from primary venue)
        df_iid["close_price"], df_iid["close_flag"] = get_close_price(iid, date)

        # Step 5: Calculate markouts at each time horizon (1ms through 5s)
        df_iid = get_ebbo_markouts(
            df_iid, ebbo, name="ebbo", execution_price="execution_price"
        )

        # Step 6: Fetch all trades to compute VWAP and TWAP
        volumes = get_trades(iid, date)

        # Step 7: Calculate VWAP (volume-weighted average price) for each execution's time window
        # Note: This uses row-by-row lambda application which can be optimized for large datasets
        # by pre-computing VWAPs for all windows, but this approach is clearer for now
        df_iid['vwap'] = df_iid.apply(lambda row: get_vwap_value(volumes, row.start_time, row.end_time), axis=1)

        # Step 8: Calculate TWAP (time-weighted average price) for each execution's time window
        df_iid['twap'] = df_iid.apply(lambda row: get_twap_value(ebbo, row.start_time, row.end_time), axis=1)

        # Return the complete TCA report
        return df_iid

    except Exception as err:
        import traceback
        print(f"generate_tca_report failed for iid={iid}, date={date}: {err}")
        traceback.print_exc()
        return None

In [ ]:
# Platform-specific implementation: Databricks Spark SQL version
# Uses Spark session to execute queries and returns pandas DataFrames
def get_iid_data(df, date, Ticker="Ticker", MIC="MIC"):
    """Databricks implementation - queries REFERENCE table using Spark SQL."""
    ticker_list = ", ".join(f"'{t}'" for t in df[Ticker].to_list())
    mic_list = ", ".join(f"'{m}'" for m in df[MIC].to_list())
    sql = f"""
        SELECT TICKER AS Ticker, MIC, OPOL, INSTRUMENT_ID AS InstrumentId, LISTING_ID AS ListingId, IS_PRIMARY AS IsPrimary
        FROM `market_data`.`vw_reference`
        WHERE TICKER IN ({ticker_list})
          AND MIC IN ({mic_list})
          AND REFERENCE_DATE = '{date}'
          AND IS_ALIVE = TRUE
    """
    ref = spark.sql(sql).toPandas()
    df_iid = df.merge(
        ref[["Ticker", "MIC", "OPOL", "InstrumentId"]],
        left_on=[Ticker, MIC],
        right_on=["Ticker", "MIC"],
        how="left",
    )
    return df_iid, ref

In [ ]:
def get_ebbo(iid, date):
    ref_iid = ref[ref.InstrumentId == iid]
    if ref_iid.empty:
        return pd.DataFrame()
    primary = ref_iid[ref_iid.IsPrimary]
    primary_lid = primary.ListingId.iloc[0]

    sql = f"""
        SELECT
            EVENT_TIMESTAMP AS event_timestamp,
            BID_PRICE1 AS ebb_price,
            ASK_PRICE1 AS ebo_price
        FROM `market_data`.`vw_l2_equity_cbbo`
        WHERE LISTING_ID = {primary_lid}
          AND TRADE_DATE = '{date}'
        ORDER BY EVENT_TIMESTAMP
    """
    ebbo = spark.sql(sql).toPandas()
    if ebbo.empty:
        return ebbo
    ebbo["mid_price"] = 0.5 * (ebbo.ebo_price + ebbo.ebb_price)
    return ebbo[["event_timestamp", "ebb_price", "ebo_price", "mid_price"]]

In [ ]:
# Platform-specific implementation: Databricks Spark SQL version
# Queries ANALYTICS table for closing and last trade prices via Spark
def get_close_price(iid, date):
    """Databricks implementation - queries ANALYTICS table for close/last prices."""
    ref_iid = ref[ref.InstrumentId == iid]
    primary_lid = ref_iid[ref_iid.IsPrimary].ListingId.iloc[0]
    sql = f"""
        SELECT CLOSE, LAST
        FROM `derived_data`.`vw_analytics`
        WHERE LISTING_ID = {primary_lid}
          AND TRADING_DATE = '{date}'
    """
    df = spark.sql(sql).toPandas()
    if df.empty:
        return None, False
    close_price = df.iloc[0].get("CLOSE")
    last_price = df.iloc[0].get("LAST")
    if close_price in (0, None) and last_price not in (0, None):
        return last_price, False
    return close_price, True

In [ ]:
# Platform-specific implementation: Databricks Spark SQL version
# Queries TRADES table across all listings for the instrument via Spark
def get_trades(iid, date):
    """Databricks implementation - queries TRADES table across all venues via Spark SQL."""
    lids = ref[ref.InstrumentId == iid].ListingId.unique().tolist()
    if not lids:
        return pd.DataFrame()
    lid_list = ", ".join(str(l) for l in lids)
    sql = f"""
        SELECT
            TRADE_TIMESTAMP AS event_timestamp,
            PRICE AS price,
            SIZE AS execution_size,
            BMLL_TRADE_TYPE AS bmll_trade_type,
            MARKET_STATE AS market_state
        FROM `market_data`.`vw_trades`
        WHERE LISTING_ID IN ({lid_list})
          AND TRADE_DATE = '{date}'
        ORDER BY TRADE_TIMESTAMP
    """
    return spark.sql(sql).toPandas()


#
## Lets generate some mock data!

In [ ]:


# Sample DataFrame with start_time and end_time included directly in the dictionary
orders_data = pd.DataFrame([
    {'Ticker': 'VOD', 'MIC': 'XLON', 'start_time': '2022-11-04 09:59:47.866947613', 'execution_price': 100.23, 'side': 1, 'end_time': '2022-11-04 10:45:47.866947613'},  # 46 minutes after start_time
    {'Ticker': 'VOD', 'MIC': 'XLON', 'start_time': '2022-11-04 10:59:47.866947613', 'execution_price': 100.4, 'side': 1,  'end_time': '2022-11-04 12:07:47.866947613'},  # 1 hour 8 minutes after start_time
    {'Ticker': 'VOD', 'MIC': 'XLON', 'start_time': '2022-11-04 11:59:47.866947613', 'execution_price': 101.55, 'side': 2, 'end_time': '2022-11-04 13:10:47.866947613'},  # 1 hour 11 minutes after start_time
    {'Ticker': 'VOD', 'MIC': 'XLON', 'start_time': '2022-11-04 12:59:47.866947613', 'execution_price': 100.24, 'side': 1, 'end_time': '2022-11-04 13:48:47.866947613'},  # 49 minutes after start_time
    {'Ticker': 'VOD', 'MIC': 'XLON', 'start_time': '2022-11-04 13:59:47.866947613', 'execution_price': 102.03, 'side': 2, 'end_time': '2022-11-04 15:09:47.866947613'}   # 1 hour 10 minutes after start_time
])

orders_data

##
## Add our bmll instrument identifier

In [ ]:
# here we add our bmll instrument identifier and get our reference data
custom_orders, ref = get_iid_data(orders_data, "2022-11-04")
custom_orders

# 
## Generate our TCA data

In [ ]:
tca_report = generate_tca_report(121429, "2022-11-04")
tca_report

# 
## Lets visualise the Data!

In [ ]:
if tca_report is not None:
    px.histogram(
        10000
        * abs(tca_report.execution_price - tca_report.arrival_price)
        / tca_report.arrival_price,
        title="Histogram: Bps Difference between execution price and arrival price",
        template="plotly_white",
    )

# 
## Running in parallel

### Batch Processing

For processing multiple instruments, the distributed version parallelizes TCA computation:
- **Notebook**: Uses Spark distributed computing for speed
- **Snowflake/Databricks**: Processes sequentially (SQL engines already optimized)

In [ ]:
def generate_tca_report_distributed(df, date):
    """
    Generate TCA reports for multiple instruments (Snowflake/Databricks).

    Processes each instrument sequentially. Returns only successful results,
    filtering out None values from failed instruments.

    Parameters:
        df (pd.DataFrame): DataFrame with InstrumentId column
        date (str): Trading date in YYYY-MM-DD format

    Returns:
        pd.DataFrame: Concatenated TCA reports for all successfully processed instruments
                     (empty DataFrame if all instruments fail)
    """
    res = [generate_tca_report(iid, date) for iid in df.InstrumentId.unique()]
    res = [r for r in res if r is not None]
    if not res:
        return pd.DataFrame()
    return pd.concat(res, ignore_index=True, copy=False)

In [ ]:

# Sample DataFrame with start_time, end_time, and additional tickers
orders_data = pd.DataFrame([
    {'Ticker': 'VOD', 'MIC': 'XLON', 'start_time': '2022-11-04 09:59:47.866947613', 'execution_price': 100.23, 'side': 1, 
     'end_time': '2022-11-04 10:45:47.866947613'},  # 46 minutes after start_time
    {'Ticker': 'VOD', 'MIC': 'XLON', 'start_time': '2022-11-04 10:59:47.866947613', 'execution_price': 100.4, 'side': 1, 
     'end_time': '2022-11-04 12:07:47.866947613'},  # 1 hour 8 minutes after start_time
    {'Ticker': 'VOD', 'MIC': 'XLON', 'start_time': '2022-11-04 11:59:47.866947613', 'execution_price': 101.55, 'side': 2, 
     'end_time': '2022-11-04 13:10:47.866947613'},  # 1 hour 11 minutes after start_time
    {'Ticker': 'VOD', 'MIC': 'XLON', 'start_time': '2022-11-04 12:59:47.866947613', 'execution_price': 100.24, 'side': 1, 
     'end_time': '2022-11-04 13:48:47.866947613'},  # 49 minutes after start_time
    {'Ticker': 'VOD', 'MIC': 'XLON', 'start_time': '2022-11-04 13:59:47.866947613', 'execution_price': 102.03, 'side': 2, 
     'end_time': '2022-11-04 15:09:47.866947613'},  # 1 hour 10 minutes after start_time
    {'Ticker': 'BMW', 'MIC': 'XETR', 'start_time': '2022-11-04 09:59:47.866947613', 'execution_price': 75.10, 'side': 1, 
     'end_time': '2022-11-04 11:05:47.866947613'},  # 1 hour 6 minutes after start_time
    {'Ticker': 'BMW', 'MIC': 'XETR', 'start_time': '2022-11-04 10:59:47.866947613', 'execution_price': 76.40, 'side': 2, 
     'end_time': '2022-11-04 12:10:47.866947613'},  # 1 hour 11 minutes after start_time
    {'Ticker': 'BMW', 'MIC': 'XETR', 'start_time': '2022-11-04 11:59:47.866947613', 'execution_price': 77.20, 'side': 1, 
     'end_time': '2022-11-04 13:15:47.866947613'},  # 1 hour 16 minutes after start_time
    {'Ticker': 'INGA', 'MIC': 'XAMS', 'start_time': '2022-11-04 10:59:47.866947613', 'execution_price': 13.80, 'side': 2, 
     'end_time': '2022-11-04 12:08:47.866947613'},  # 1 hour 9 minutes after start_time
    {'Ticker': 'INGA', 'MIC': 'XAMS', 'start_time': '2022-11-04 11:59:47.866947613', 'execution_price': 14.30, 'side': 1, 
     'end_time': '2022-11-04 13:06:47.866947613'},  # 1 hour 7 minutes after start_time
    {'Ticker': 'INGA', 'MIC': 'XAMS', 'start_time': '2022-11-04 12:59:47.866947613', 'execution_price': 14.10, 'side': 1, 
     'end_time': '2022-11-04 14:05:47.866947613'}   # 1 hour 6 minutes after start_time
])

orders_data

In [ ]:
# here we add our bmll instrument identifier and get our reference data
custom_orders, ref = get_iid_data(orders_data, "2022-11-04")
custom_orders

In [ ]:
tca_result = generate_tca_report_distributed(custom_orders, "2022-11-04")
tca_result